# MVT-RAG vs Baselines on QASPER Scientific QA

This notebook demonstrates **MVT-RAG** (Marginal Value Theorem-based Retrieval-Augmented Generation),
an ecology-inspired stopping criterion for adaptive document retrieval.

**Core idea**: The Marginal Value Theorem (MVT) from foraging ecology says an animal should leave a
food patch when the marginal gain from the current patch falls below the average yield across all
patches in the environment (`G_env`). Applied to RAG: stop retrieving from the current document
section when `marginal_gain = relevance × novelty < G_env`, then switch to the next most promising
section.

**Dataset**: [QASPER](https://huggingface.co/datasets/allenai/qasper) — scientific QA on NLP papers.

**What this notebook does**:
1. Loads pre-computed results from the full experiment (100 papers, 223 questions)
2. Demonstrates the retrieval algorithms on a small live example (1 paper, 2 questions)
3. Compares MVT-RAG against 8 baselines: top-k dense, BM25, confidence-threshold, ablation, no-RAG

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Not pre-installed on Colab
_pip('rank-bm25==0.2.2')
_pip('sentence-transformers==3.3.1')
_pip('loguru==0.7.3')
_pip('psutil==6.1.1')

# Pre-installed on Colab — install locally only to match Colab versions
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scikit-learn==1.6.1', 'matplotlib==3.10.0',
         'datasets==4.0.0', 'tqdm==4.67.3')

In [ ]:
import json
import math
import os
import sys
from pathlib import Path
from typing import Any

import numpy as np
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-7ae548-efficiency-at-a-cost-applying-the-margin/main/round-1/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
examples = data['datasets'][0]['examples']
summary_stats = data['metadata']['summary_stats']
print(f"Loaded {len(examples)} pre-computed examples")
print(f"Methods: {data['metadata']['retrieval_methods']}")

## Configuration

Demo parameters — set to minimum to run fast. The full experiment used `N_PAPERS=100`, `N_QUESTIONS=223`.

In [ ]:
# Demo config — minimum values for fast execution
N_PAPERS = 1          # number of QASPER papers to load for live demo (full run: 100)
N_QUESTIONS = 2       # max questions per paper to process live (full run: all)
MAX_CHUNKS_CAP = 20   # cap on chunks retrieved by MVT (same as full run)
EMBED_BATCH = 32      # embedding batch size (full run: 128)
METHODS = [
    "mvt_rag", "mvt_noenv",
    "topk_3", "topk_5", "topk_10",
    "bm25_5", "thresh_0.3", "thresh_0.5",
    "no_rag",
]

## Data Loading — QASPER

QASPER stores full text in a **columnar dict** format:
`{section_name: [str, ...], paragraphs: [[str, ...], ...]}`. We parse this into a list of sections,
each with their paragraph chunks.

In [ ]:
def load_qasper(n_papers: int) -> list[dict]:
    from datasets import load_dataset
    print("Loading QASPER validation split...")
    ds = load_dataset("allenai/qasper", split="validation", trust_remote_code=True)
    papers = list(ds)[:n_papers]
    print(f"Loaded {len(papers)} papers")
    return papers

def parse_paper(paper: dict) -> list[dict]:
    """Parse paper into sections with chunks.

    QASPER full_text is a columnar dict:
      {"section_name": [str, ...], "paragraphs": [[str, ...], ...]}
    """
    sections = []
    ft = paper.get("full_text") or {}
    if isinstance(ft, dict):
        names = ft.get("section_name") or []
        paragraphs_list = ft.get("paragraphs") or []
        for name, paras in zip(names, paragraphs_list):
            name = (name or "unknown").strip() or "unknown"
            paras = paras or []
            chunks = [p.strip() for p in paras if isinstance(p, str) and p.strip()]
            if chunks:
                sections.append({"name": name, "chunks": chunks})
    elif isinstance(ft, list):
        for section in ft:
            name = (section.get("section_name") or "unknown").strip() or "unknown"
            paras = section.get("paragraphs") or []
            chunks = [p.strip() for p in paras if isinstance(p, str) and p.strip()]
            if chunks:
                sections.append({"name": name, "chunks": chunks})
    if not sections:
        abstract = paper.get("abstract", "")
        if abstract:
            sections.append({"name": "abstract", "chunks": [abstract]})
    return sections

papers = load_qasper(N_PAPERS)
print(f"\nPaper title: {papers[0].get('title', 'N/A')[:80]}")

## Embedding

We use `all-MiniLM-L6-v2` (22MB) to compute dense embeddings for all chunks in a paper.
Embeddings are L2-normalized so cosine similarity = dot product.

In [ ]:
_embed_model = None

def get_embed_model():
    global _embed_model
    if _embed_model is None:
        from sentence_transformers import SentenceTransformer
        print("Loading SentenceTransformer all-MiniLM-L6-v2...")
        _embed_model = SentenceTransformer("all-MiniLM-L6-v2")
        print("Embedding model loaded")
    return _embed_model

def embed_texts(texts: list[str]) -> np.ndarray:
    model = get_embed_model()
    return model.encode(texts, batch_size=EMBED_BATCH, show_progress_bar=False, normalize_embeddings=True)

def embed_paper(sections: list[dict]) -> tuple[list[tuple[str, str]], np.ndarray]:
    """Returns (chunk_meta, embeddings). chunk_meta = list of (section_name, chunk_text)."""
    chunk_meta = [(s["name"], c) for s in sections for c in s["chunks"]]
    texts = [c for _, c in chunk_meta]
    embs = embed_texts(texts)
    return chunk_meta, embs

# Pre-load the model
get_embed_model()

## Retrieval Methods

### MVT-RAG Algorithm

Inspired by optimal foraging theory:
1. Compute `G_env = mean(per-section best similarity)` — the environmental average patch quality
2. Visit sections in order of decreasing potential
3. For each chunk: compute `G_t = relevance × novelty`
4. **Stop** and switch sections when `G_t < G_env`

Where `relevance = cosine_sim(query, chunk)` and `novelty = 1 - max_sim(chunk, already_retrieved)`.

In [ ]:
def _build_sec_map(chunk_meta: list[tuple[str, str]]) -> dict[str, list[int]]:
    sec_map: dict[str, list[int]] = {}
    for i, (sname, _) in enumerate(chunk_meta):
        sec_map.setdefault(sname, []).append(i)
    return sec_map

def mvt_rag(
    query_emb: np.ndarray,
    chunk_embs: np.ndarray,
    chunk_meta: list[tuple[str, str]],
) -> tuple[list[str], float]:
    """MVT-RAG: switch sections based on ecology-derived G_env threshold."""
    sec_map = _build_sec_map(chunk_meta)
    if not sec_map:
        return [], 0.0

    # G_env: mean of per-section best similarity → average patch quality
    sec_best_sims = []
    sec_potential: dict[str, float] = {}
    for sname, idxs in sec_map.items():
        sims = cosine_similarity(query_emb.reshape(1, -1), chunk_embs[idxs])[0]
        best = float(np.max(sims))
        sec_best_sims.append(best)
        sec_potential[sname] = best
    G_env = float(np.mean(sec_best_sims))

    retrieved: list[tuple[str, np.ndarray]] = []
    visited: set[str] = set()

    while len(retrieved) < MAX_CHUNKS_CAP:
        remaining = {s: p for s, p in sec_potential.items() if s not in visited}
        if not remaining:
            break
        cur_sec = max(remaining, key=remaining.get)  # type: ignore
        visited.add(cur_sec)

        ret_embs = [r[1] for r in retrieved]

        for idx in sec_map[cur_sec]:
            chunk_emb = chunk_embs[idx]
            chunk_text = chunk_meta[idx][1]

            q_sim = float(cosine_similarity(query_emb.reshape(1, -1), chunk_emb.reshape(1, -1))[0][0])

            if ret_embs:
                ret_arr = np.stack(ret_embs)
                max_ret_sim = float(np.max(cosine_similarity(chunk_emb.reshape(1, -1), ret_arr)[0]))
                novelty = 1.0 - max_ret_sim
            else:
                novelty = 1.0

            G_t = q_sim * novelty

            if G_t < G_env and retrieved:
                break  # switch to next section

            retrieved.append((chunk_text, chunk_emb))
            ret_embs.append(chunk_emb)

    return [r[0] for r in retrieved], G_env

def mvt_noenv_rag(
    query_emb: np.ndarray,
    chunk_embs: np.ndarray,
    chunk_meta: list[tuple[str, str]],
    threshold: float = 0.5,
) -> list[str]:
    """MVT ablation: fixed threshold instead of ecology-derived G_env."""
    sec_map = _build_sec_map(chunk_meta)
    if not sec_map:
        return []

    sec_potential: dict[str, float] = {}
    for sname, idxs in sec_map.items():
        sims = cosine_similarity(query_emb.reshape(1, -1), chunk_embs[idxs])[0]
        sec_potential[sname] = float(np.max(sims))

    retrieved: list[tuple[str, np.ndarray]] = []
    visited: set[str] = set()

    while len(retrieved) < MAX_CHUNKS_CAP:
        remaining = {s: p for s, p in sec_potential.items() if s not in visited}
        if not remaining:
            break
        cur_sec = max(remaining, key=remaining.get)  # type: ignore
        visited.add(cur_sec)

        ret_embs = [r[1] for r in retrieved]

        for idx in sec_map[cur_sec]:
            chunk_emb = chunk_embs[idx]
            chunk_text = chunk_meta[idx][1]

            q_sim = float(cosine_similarity(query_emb.reshape(1, -1), chunk_emb.reshape(1, -1))[0][0])

            if ret_embs:
                ret_arr = np.stack(ret_embs)
                max_ret_sim = float(np.max(cosine_similarity(chunk_emb.reshape(1, -1), ret_arr)[0]))
                novelty = 1.0 - max_ret_sim
            else:
                novelty = 1.0

            G_t = q_sim * novelty

            if G_t < threshold and retrieved:
                break

            retrieved.append((chunk_text, chunk_emb))
            ret_embs.append(chunk_emb)

    return [r[0] for r in retrieved]

def topk_dense(
    query_emb: np.ndarray,
    chunk_embs: np.ndarray,
    chunk_meta: list[tuple[str, str]],
    k: int,
) -> list[str]:
    sims = cosine_similarity(query_emb.reshape(1, -1), chunk_embs)[0]
    idxs = np.argsort(sims)[::-1][:k]
    return [chunk_meta[i][1] for i in idxs]

def bm25_retrieval(
    query: str,
    chunk_meta: list[tuple[str, str]],
    k: int = 5,
) -> list[str]:
    corpus = [c.split() for _, c in chunk_meta]
    if not corpus:
        return []
    bm25 = BM25Okapi(corpus)
    scores = bm25.get_scores(query.split())
    idxs = np.argsort(scores)[::-1][:k]
    return [chunk_meta[i][1] for i in idxs]

def threshold_retrieval(
    query_emb: np.ndarray,
    chunk_embs: np.ndarray,
    chunk_meta: list[tuple[str, str]],
    threshold: float,
) -> list[str]:
    sims = cosine_similarity(query_emb.reshape(1, -1), chunk_embs)[0]
    order = np.argsort(sims)[::-1]
    chunks = [chunk_meta[i][1] for i in order if sims[i] >= threshold]
    if not chunks:
        chunks = [chunk_meta[order[0]][1]]  # at least one
    return chunks[:MAX_CHUNKS_CAP]

print("Retrieval functions defined.")

## Evaluation Metrics

- **Token F1**: overlap between predicted and gold answer tokens
- **Oracle retrieval F1**: fraction of gold extractive spans that appear in the retrieved chunks (measures retrieval quality, independent of LLM)

In [ ]:
def oracle_retrieval_f1(retrieved_chunks: list[str], gold_spans: list[str]) -> float:
    """Check if gold extractive spans appear in retrieved chunks."""
    if not gold_spans or not retrieved_chunks:
        return 0.0
    retrieved_text = " ".join(retrieved_chunks).lower()
    hits = sum(1 for span in gold_spans if span.lower().strip() in retrieved_text)
    return hits / len(gold_spans)

def token_f1(pred: str, gold: str) -> float:
    pred_toks = set(pred.lower().split())
    gold_toks = set(gold.lower().split())
    if not pred_toks or not gold_toks:
        return 0.0
    common = pred_toks & gold_toks
    if not common:
        return 0.0
    p = len(common) / len(pred_toks)
    r = len(common) / len(gold_toks)
    return 2 * p * r / (p + r)

def best_f1(pred: str, gold_answers: list[str]) -> float:
    return max((token_f1(pred, g) for g in gold_answers), default=0.0)

print("Metrics defined.")

## Live Demo — Retrieval on a Real QASPER Paper

We parse a paper, embed all chunks, then run all retrieval methods on the first few questions.
No LLM calls needed — we just compare what each method retrieves.

In [ ]:
paper = papers[0]
sections = parse_paper(paper)
chunk_meta, chunk_embs = embed_paper(sections)

print(f"Paper: {paper.get('title', 'N/A')[:80]}")
print(f"Sections: {len(sections)}, Chunks: {len(chunk_meta)}")

# Extract questions
qas_col = paper.get("qas") or {}
questions_list = qas_col.get("question") or [] if isinstance(qas_col, dict) else []
answers_col = qas_col.get("answers") or [] if isinstance(qas_col, dict) else []

demo_results = []
for question, answers_for_q in list(zip(questions_list, answers_col))[:N_QUESTIONS]:
    question = (question or "").strip()
    if not question:
        continue

    # Parse gold answers
    gold_spans = []
    ans_list = answers_for_q.get("answer") or [] if isinstance(answers_for_q, dict) else []
    for ans in ans_list:
        if not isinstance(ans, dict) or ans.get("unanswerable"):
            continue
        for span in ans.get("extractive_spans") or []:
            if span and span.strip():
                gold_spans.append(span.strip())

    query_emb = embed_texts([question])[0]
    mvt_chunks, g_env = mvt_rag(query_emb, chunk_embs, chunk_meta)

    method_chunks = {
        "mvt_rag": mvt_chunks,
        "mvt_noenv": mvt_noenv_rag(query_emb, chunk_embs, chunk_meta),
        "topk_3": topk_dense(query_emb, chunk_embs, chunk_meta, 3),
        "topk_5": topk_dense(query_emb, chunk_embs, chunk_meta, 5),
        "topk_10": topk_dense(query_emb, chunk_embs, chunk_meta, 10),
        "bm25_5": bm25_retrieval(question, chunk_meta, 5),
        "thresh_0.3": threshold_retrieval(query_emb, chunk_embs, chunk_meta, 0.3),
        "thresh_0.5": threshold_retrieval(query_emb, chunk_embs, chunk_meta, 0.5),
        "no_rag": [],
    }

    oracle_f1s = {m: oracle_retrieval_f1(chunks, gold_spans) for m, chunks in method_chunks.items()}
    n_chunks = {m: len(chunks) for m, chunks in method_chunks.items()}

    demo_results.append({
        "question": question,
        "g_env": g_env,
        "gold_spans": gold_spans[:3],
        "oracle_f1": oracle_f1s,
        "n_chunks": n_chunks,
    })

    print(f"\nQ: {question[:80]}")
    print(f"  G_env={g_env:.3f}  (MVT threshold for section switching)")
    for m in METHODS:
        print(f"  {m:15s}: {n_chunks[m]:2d} chunks, oracle_f1={oracle_f1s[m]:.2f}")

## Bootstrap Statistical Test

Used in the full experiment to test whether MVT-RAG is significantly different from the topk_5 baseline.

In [ ]:
def bootstrap_p(a: list[float], b: list[float], n: int = 5000) -> float:
    """Bootstrap p-value: P(mean(a) <= mean(b)) under resampling."""
    if not a or not b or len(a) != len(b):
        return float("nan")
    arr_a = np.array(a)
    arr_b = np.array(b)
    rng = np.random.default_rng(42)
    n_s = len(arr_a)
    obs_diff = np.mean(arr_a) - np.mean(arr_b)
    boot_diffs = []
    for _ in range(n):
        idx = rng.integers(0, n_s, n_s)
        boot_diffs.append(np.mean(arr_a[idx]) - np.mean(arr_b[idx]))
    return float(np.mean(np.array(boot_diffs) <= 0))

print("Bootstrap test defined.")

## Full Experiment Results (Pre-computed)

Results from the full run: **100 QASPER validation papers, 223 answerable questions**.
LLM: `meta-llama/llama-3.1-8b-instruct` via OpenRouter. Total cost: ~$0.40.

Key finding: MVT-RAG retrieves far fewer chunks (1.3 vs 5) but achieves lower F1 than topk_5,
indicating the G_env threshold is too aggressive at these scales.

In [ ]:
# Print summary table from pre-computed results
print(f"{'Method':<15} {'F1':>6} {'±std':>6} {'Oracle':>7} {'Chunks':>7} {'p_vs_topk5':>11}")
print("-" * 60)
for method in METHODS:
    s = summary_stats.get(method, {})
    p = s.get('p_vs_topk5_f1', float('nan'))
    p_str = f"{p:.3f}" if not math.isnan(p) else "  ref"
    print(f"{method:<15} {s.get('mean_f1',0):.3f}  {s.get('std_f1',0):.3f}  "
          f"{s.get('mean_oracle_retrieval_f1',0):.3f}  {s.get('mean_chunks',0):5.1f}  {p_str:>11}")

## Visualization

Two plots comparing all methods:
1. **F1 score vs. average chunks retrieved** — efficiency/quality tradeoff
2. **Oracle retrieval F1** — how often gold spans appear in the retrieved context

In [ ]:
methods = METHODS
mean_f1     = [summary_stats[m]['mean_f1'] for m in methods]
std_f1      = [summary_stats[m]['std_f1'] for m in methods]
oracle_f1   = [summary_stats[m]['mean_oracle_retrieval_f1'] for m in methods]
mean_chunks = [summary_stats[m]['mean_chunks'] for m in methods]

# Color: MVT methods in blue, baselines in gray, no_rag in red
colors = []
for m in methods:
    if m.startswith('mvt'):
        colors.append('#2563eb')  # blue
    elif m == 'no_rag':
        colors.append('#dc2626')  # red
    else:
        colors.append('#6b7280')  # gray

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Plot 1: F1 vs Chunks (efficiency/quality tradeoff) ---
ax = axes[0]
for i, m in enumerate(methods):
    ax.scatter(mean_chunks[i], mean_f1[i], color=colors[i], s=90, zorder=5)
    ax.annotate(m, (mean_chunks[i], mean_f1[i]),
                textcoords='offset points', xytext=(5, 3), fontsize=8)

ax.set_xlabel('Avg. Chunks Retrieved', fontsize=11)
ax.set_ylabel('Mean Token F1', fontsize=11)
ax.set_title('Quality vs. Retrieval Budget', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
legend_handles = [
    mpatches.Patch(color='#2563eb', label='MVT methods'),
    mpatches.Patch(color='#6b7280', label='Baselines'),
    mpatches.Patch(color='#dc2626', label='No RAG'),
]
ax.legend(handles=legend_handles, fontsize=9)

# --- Plot 2: Oracle Retrieval F1 by method ---
ax2 = axes[1]
bars = ax2.bar(range(len(methods)), oracle_f1, color=colors, edgecolor='white', linewidth=0.5)
ax2.set_xticks(range(len(methods)))
ax2.set_xticklabels(methods, rotation=40, ha='right', fontsize=8)
ax2.set_ylabel('Oracle Retrieval F1', fontsize=11)
ax2.set_title('Oracle F1 — Gold Span Recall\n(fraction of gold spans in retrieved context)', fontsize=11, fontweight='bold')
ax2.grid(True, axis='y', alpha=0.3)
for bar, val in zip(bars, oracle_f1):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('mvt_rag_results.png', dpi=120, bbox_inches='tight')
plt.show()
print("\nKey finding: MVT-RAG retrieves fewer chunks (efficiency) but at a quality cost.")
print(f"  MVT-RAG:  F1={mean_f1[0]:.3f}, chunks={mean_chunks[0]:.1f}, oracle={oracle_f1[0]:.3f}")
print(f"  topk_5:   F1={mean_f1[3]:.3f}, chunks={mean_chunks[3]:.1f}, oracle={oracle_f1[3]:.3f}")